# Hướng 1 — Selective Prediction / Abstention
## “Dạy AI biết khi nào nên im lặng” (trên Gemma 4 E4B)

**Ý tưởng:** MedRegA luôn chỉ đại một vùng kể cả khi không chắc. Ta thêm cơ chế để AI **từ chối trả lời** (im lặng) khi độ tin cậy thấp → an toàn lâm sàng hơn.

**Notebook này làm gì (theo đúng đề cương):**
1. Đo 3 nguồn “độ tin cậy” của AI khi nó chỉ một vùng:
   - `logprob`: xác suất trung bình của các token toạ độ AI sinh ra.
   - `spatial`: độ nhất quán không gian khi cho AI trả lời lại nhiều lần (N lần).
   - `selfconf`: hỏi thẳng AI “bạn chắc bao nhiêu %”.
2. **Thí nghiệm cổng (gate):** kiểm tra 3 tín hiệu này có tương quan với độ đúng thật (IoU) không. Nếu không → báo cáo negative result (vẫn là đóng góp).
3. Vẽ **đường risk–coverage** + tính **selective IoU**: trả lời ít hơn nhưng chắc hơn.

**Chạy ngay được:** để `USE_MOCK = True` (mô phỏng, không cần GPU). Khi có model thật, đổi `USE_MOCK = False` và điền phần tải model.


## 1. Cài đặt thư viện
Chế độ MOCK chỉ cần numpy/scipy/sklearn/matplotlib. Phần model thật mới cần transformers/peft/bitsandbytes.

In [ ]:
# Cho chế độ MOCK (đủ để chạy thử pipeline):
%pip install -q numpy scipy scikit-learn matplotlib

# Khi dùng model thật (bỏ comment):
# %pip install -q transformers accelerate peft bitsandbytes pillow torch

## 2. Cấu hình

In [ ]:
USE_MOCK   = False                     # True: mô phỏng (không cần GPU); False: dùng model thật
MODEL_ID   = 'google/gemma-4-e4b-it'   # <-- ĐIỀN id thật khi có quyền truy cập (hoặc đổi VLM khác)
N_SAMPLES  = 5                         # số lần sample để đo độ nhất quán không gian
COORD_SCALE = 1000                     # toạ độ chuẩn hoá [0,1000) giống MedRegA
IOU_CORRECT = 0.5                      # ngưỡng coi là 'đúng'
SEED = 0

import random, numpy as np
random.seed(SEED); np.random.seed(SEED)
print('Cấu hình xong. USE_MOCK =', USE_MOCK)

## 3. Hàm tiện ích: đọc box từ text + tính IoU
AI xuất vùng dưới dạng chữ, ví dụ `<box>[388,270,956,1281]</box>`. Ta tách 4 số ra, và tính IoU (độ chồng lấn giữa box AI đoán và box đúng).

In [ ]:
import re

def parse_box(text):
    '''Lấy 4 số nguyên đầu tiên trong chuỗi -> [x1,y1,x2,y2]; None nếu không đủ.'''
    nums = re.findall(r'\d+', str(text))
    if len(nums) < 4:
        return None
    return [int(x) for x in nums[:4]]

def iou(b1, b2):
    '''IoU giữa 2 box [x1,y1,x2,y2]. 1.0 = trùng khít, 0.0 = không chồng.'''
    if b1 is None or b2 is None:
        return 0.0
    xa, ya = max(b1[0], b2[0]), max(b1[1], b2[1])
    xb, yb = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    a1 = max(0, b1[2]-b1[0]) * max(0, b1[3]-b1[1])
    a2 = max(0, b2[2]-b2[0]) * max(0, b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

# kiểm tra nhanh
print(parse_box('<box>[10, 20, 110, 220]</box>'))
print('IoU khít:', iou([0,0,100,100], [0,0,100,100]))
print('IoU lệch:', round(iou([0,0,100,100], [50,50,150,150]), 3))

## 4. Tải model — hoặc MOCK
- **MOCK:** một model giả mô phỏng: mẫu càng “khó” thì box càng lệch và độ tin cậy càng thấp (để mày thấy pipeline + tương quan hoạt động).
- **Model thật:** điền phần `AutoModelForImageTextToText`. Cần tự viết hàm `predict()` trả về `box` + `avg_logprob`, và `self_confidence()`.

In [ ]:
if USE_MOCK:
    class MockVLM:
        '''Model giả: difficulty d in [0,1] điều khiển độ lệch box & độ tin cậy.'''
        def predict(self, sample, temperature=0.0):
            d  = sample['difficulty']
            gt = sample['gt_box']
            jitter = 1.0 if temperature > 0 else 0.4   # sample nóng -> dao động nhiều hơn
            scale = COORD_SCALE * (0.02 + 0.30 * d) * jitter
            noise = np.random.normal(0, scale, 4)
            box = [int(np.clip(gt[i] + noise[i], 0, COORD_SCALE - 1)) for i in range(4)]
            box = [min(box[0], box[2]), min(box[1], box[3]), max(box[0], box[2]), max(box[1], box[3])]
            avg_logprob = float(np.clip(-0.1 - 2.5 * d + np.random.normal(0, 0.1), -5, 0))
            return {'box': box, 'avg_logprob': avg_logprob}
        def self_confidence(self, sample):
            return float(np.clip(1.0 - sample['difficulty'] + np.random.normal(0, 0.05), 0, 1))
    model = MockVLM()
    print('>> Chạy chế độ MOCK (mô phỏng) — không cần GPU/model thật.')
else:
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model_hf  = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
    # TODO (điền): bọc thành lớp có:
    #   .predict(sample, temperature): chạy model trên (ảnh + câu hỏi), trả {'box', 'avg_logprob'}.
    #        - box  = parse_box(text sinh ra)
    #        - avg_logprob = trung bình log-prob của các token toạ độ (dùng generate(output_scores=True))
    #   .self_confidence(sample): hỏi lại model 'bạn chắc bao nhiêu %' rồi map về [0,1].
    raise NotImplementedError('Điền phần gọi model thật ở đây (xem chú thích trên).')

## 5. Dữ liệu
MOCK dùng dữ liệu giả (ảnh + câu hỏi + box đúng). **Thay bằng dữ liệu vùng y khoa thật** (vd subset MIMIC-CXR + box Chest ImaGenome): mỗi mẫu cần `gt_box` và phần ảnh/câu hỏi cho model thật.

In [ ]:
def make_demo_dataset(n=80):
    data = []
    for i in range(n):
        d  = float(np.random.rand())              # độ khó ẩn (chỉ MOCK dùng)
        x1 = np.random.randint(0, 600); y1 = np.random.randint(0, 600)
        w  = np.random.randint(80, 300); h  = np.random.randint(80, 300)
        gt = [x1, y1, min(x1 + w, 999), min(y1 + h, 999)]
        data.append({'id': i, 'question': 'Hãy định vị gan và xuất bounding box.',
                     'image_path': None, 'gt_box': gt, 'difficulty': d})
    return data

dataset = make_demo_dataset()
print('Số mẫu:', len(dataset))
print('Mẫu đầu:', dataset[0])

## 6. Ba nguồn “độ tin cậy”

In [ ]:
def conf_logprob(model, sample):
    '''Tín hiệu 1: xác suất token toạ độ. Trả về (box dự đoán, độ tin cậy).'''
    out = model.predict(sample, temperature=0.0)
    return out['box'], out['avg_logprob']

def conf_spatial_consistency(model, sample, n=N_SAMPLES):
    '''Tín hiệu 2: cho trả lời lại n lần, đo IoU trung bình từng cặp.
       Nhất quán cao = tự tin cao.'''
    boxes = [model.predict(sample, temperature=0.7)['box'] for _ in range(n)]
    pair = [iou(boxes[i], boxes[j]) for i in range(len(boxes)) for j in range(i+1, len(boxes))]
    return float(np.mean(pair)) if pair else 0.0

def conf_self(model, sample):
    '''Tín hiệu 3: hỏi thẳng model độ tự tin.'''
    return model.self_confidence(sample)

print('Đã định nghĩa 3 tín hiệu tin cậy.')

## 7. Thí nghiệm CỔNG (gate) — tín hiệu tin cậy có dự báo được độ đúng không?
Đây là bước **bắt buộc** trước khi làm abstention. Nếu cả 3 tín hiệu đều **không** tương quan với IoU thật → kết luận negative (vẫn đáng báo cáo).

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

rows = []
for s in dataset:
    box, lp = conf_logprob(model, s)
    rows.append({
        'iou':      iou(box, s['gt_box']),
        'logprob':  lp,
        'spatial':  conf_spatial_consistency(model, s),
        'selfconf': conf_self(model, s),
    })

ious    = np.array([r['iou'] for r in rows])
correct = (ious >= IOU_CORRECT).astype(int)
print(f'IoU trung bình: {ious.mean():.3f} | tỉ lệ đúng (IoU>={IOU_CORRECT}): {correct.mean():.2%}\n')
print(f"{'Tín hiệu':10s}  {'Spearman vs IoU':>16s}  {'AUROC đúng/sai':>16s}")
for key in ['logprob', 'spatial', 'selfconf']:
    v = np.array([r[key] for r in rows])
    rho = spearmanr(v, ious).correlation
    auc = roc_auc_score(correct, v) if len(set(correct)) > 1 else float('nan')
    print(f'{key:10s}  {rho:>+16.3f}  {auc:>16.3f}')

**Đọc kết quả:** Spearman/AUROC càng gần 1 → tín hiệu càng dự báo tốt độ đúng (nên dùng để quyết định im lặng). Gần 0.5 (AUROC) hoặc gần 0 (Spearman) → tín hiệu vô dụng.

## 8. Đường Risk–Coverage + Selective IoU
Sắp xếp các câu theo độ tin cậy; chỉ trả lời phần tự tin nhất. **Coverage** = tỉ lệ câu chịu trả lời; trục còn lại = IoU trung bình trên phần đã trả lời. Đường đi lên khi coverage giảm = “trả lời ít hơn nhưng chắc hơn”.

In [ ]:
import matplotlib.pyplot as plt

BEST_SIGNAL = 'spatial'   # đổi sang tín hiệu có AUROC cao nhất ở bước 7
signal = np.array([r[BEST_SIGNAL] for r in rows])
order  = np.argsort(-signal)                 # tự tin cao -> thấp

cov, sel_iou = [], []
for k in range(1, len(order) + 1):
    idx = order[:k]
    cov.append(k / len(order))
    sel_iou.append(float(ious[idx].mean()))

plt.figure(figsize=(6, 4))
plt.plot(cov, sel_iou, marker='o', ms=3)
plt.gca().invert_xaxis()
plt.xlabel('Coverage (tỉ lệ câu AI chịu trả lời)')
plt.ylabel('IoU trung bình trên phần đã trả lời')
plt.title(f'Risk-Coverage (tín hiệu: {BEST_SIGNAL})')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ví dụ: nếu chỉ trả lời 70% câu tự tin nhất thì IoU trung bình là bao nhiêu
for c in [1.0, 0.8, 0.6, 0.4]:
    k = max(1, int(c * len(order)))
    print(f'Coverage {c:.0%}: selective IoU = {ious[order[:k]].mean():.3f}')

## 9. Bước tiếp theo (khi có model thật)
1. Đổi `USE_MOCK = False`, điền `predict()` / `self_confidence()` cho Gemma 4 E4B.
2. Thay `make_demo_dataset()` bằng dữ liệu vùng y khoa thật (ảnh + câu hỏi + box GT).
3. Chạy lại bước 7 → chọn tín hiệu tin cậy tốt nhất.
4. (Nâng cao) **Conformal theo từng loại ảnh** để bảo đảm coverage; thêm **cost-aware selective IoU** (phạt nặng khi định vị bừa lúc sai).

> Lưu ý: đây là proof-of-concept cơ chế trên base nhỏ, không nhằm so kè bảng SOTA.